# Animal Sound Classification

Versione aggiornata per una nuova esecuzione: il ramo waveform ricampiona a 16 kHz e usa un secondo di audio; il ramo Mel mantiene quattro secondi a 22,05 kHz. I risultati storici sono conservati in `notebooks/AnimalCNN-original.ipynb` e non sono risultati di questa versione.

Imposta `DATASET_ROOT` e `ARTIFACT_DIR` nella configurazione. Per usare Colab, monta il Drive con il dataset. In locale imposta `ANIMAL_DATASET_DIR` oppure modifica il percorso nella cella di configurazione. Installa prima le dipendenze della prossima cella (Python 3.12).

Il peso aggiuntivo della classe cane nel modello Mel conserva il fattore sperimentale originale ed è modificabile con `DOG_CLASS_WEIGHT_MULTIPLIER`; non è un valore validato su altri dataset.


In [ ]:
%pip install tensorflow==2.19.0 numpy==2.1.3 librosa==0.10.2.post1 audiomentations==0.41.0 scikit-learn==1.7.0 matplotlib==3.11.2 seaborn==0.13.2


# **Riconoscimento di Animali da Tracce Audio**

---





#Introduzione
Il seguente progetto è stato svolto in autonomia da **Angelo Romano**, studente di Ingegneria Informatica del primo anno Magistrale presso l'**Università degli studi di Roma "Tor Vergata"**.

### **Contesto**
Il progetto si focalizza sulla classificazione di tracce audio per riconoscere la specie animale (cani, gatti e delfini) che le ha prodotte.


### **Dataset**
Il dataset, fornitoci dal professore, include registrazioni audio pre-etichettate, con file audio in formato `.wav` raggruppati per specie.

### **Obiettivi**
- Sviluppare un modello che utilizzi direttamente la **forma d’onda** come input.
- Sviluppare un secondo modello che utilizzi lo **spettrogramma** come input.
- Confrontare le prestazioni dei due modelli in termini di **accuratezza**, **precisione** e **costo computazionale**.



#Setup
In questa sezione vengono importate le librerie necessarie per il progetto, tra cui Librosa per l'elaborazione dell'audio, Matplotlib e Seaborn per la visualizzazione dei dati, TensorFlow per la creazione del modello e altre librerie utili per la gestione dei file.

**Import**


In [ ]:
import os
import random
from pathlib import Path
from glob import glob

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from audiomentations import TimeStretch, PitchShift, BandPassFilter, TanhDistortion

try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')

SEED = 42
WAVEFORM_SR = 16000
WAVEFORM_SAMPLES = 16000
VALIDATION_FRACTION = 0.2
MAX_EPOCHS = 50
DOG_CLASS_WEIGHT_MULTIPLIER = 5.0

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

colab_data = '/content/drive/MyDrive/cats_dogs_dolphins'
local_data = 'data/cats_dogs_dolphins'
DATASET_ROOT = Path(os.environ.get('ANIMAL_DATASET_DIR', colab_data if Path('/content').exists() else local_data))
ARTIFACT_DIR = Path(os.environ.get('ANIMAL_OUTPUT_DIR', 'outputs'))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


###Caricamento dati
Si definiscono i percorsi dei file audio suddivisi per categoria (gatti, cani, delfini) per il training set e il test set. I file vengono raccolti automaticamente utilizzando glob, consentendo una gestione dinamica dei dati.

In [ ]:
data_paths = {
    label: sorted(glob(str(DATASET_ROOT / 'train' / label / '*.wav')))
    for label in ('cat', 'dog', 'dolphin')
}
test_data_paths = {
    label: sorted(glob(str(DATASET_ROOT / 'test' / label / '*.wav')))
    for label in ('cat', 'dog', 'dolphin')
}
for split, paths in [('train', data_paths), ('test', test_data_paths)]:
    for label, files in paths.items():
        if not files:
            raise FileNotFoundError(f'No WAV files found in {DATASET_ROOT / split / label}')
        print(f'{split}/{label}: {len(files)} recordings')


##Analisi Dataset

Viene fattaun'analisi esplorativa delle caratteristiche principali dei dati audio:

**Durata dei file audio**: Viene calcolata la distribuzione delle durate per ogni categoria.  

**Distribuzione delle frequenze medie**: Viene analizzata la risposta in frequenza media per ogni classe, utile per comprendere differenze tra specie.  

**Visualizzazione delle waveforms e degli MFCC**: Viene plottata la forma d'onda e la rappresentazione MFCC (Mel-Frequency Cepstral Coefficients) per campioni esemplificativi, fornendo un'intuizione sulle caratteristiche distintive dei segnali audio.

###Funzioni di analisi
Queste funzioni servono per estrarre le informazioni dagli audio per rappresentarle poi graficamente in diversi modi





In [ ]:
import seaborn as sns
# Funzione per calcolare la durata dei file audio
def get_audio_durations(file_paths):
    durations = []
    for file in file_paths:
        audio, sr = librosa.load(file, sr=None)
        duration = librosa.get_duration(y=audio, sr=sr)
        durations.append(duration)
    return durations

# Funzione per calcolare la distribuzione delle frequenze medie
def get_frequency_distribution(file_paths):
    mean_freqs = []
    for file in file_paths:
        audio, sr = librosa.load(file, sr=WAVEFORM_SR)
        D = np.abs(librosa.stft(audio))  # Spettrogramma
        mean_freq = np.mean(D, axis=1)
        mean_freqs.append(mean_freq)
    return np.array(mean_freqs)

# Funzione per visualizzare insieme (side by side) il boxplot delle durate e la distribuzione della frequenza media
def plot_duration_and_frequency(durations, freqs, label):
    fig, axs = plt.subplots(1, 2, figsize=(16, 6))

    sns.boxplot(x=durations, color="skyblue", ax=axs[0])


    axs[0].set_title(f"Distribuzione delle durate per {label.capitalize()}")
    axs[0].set_ylabel("Durata (secondi)")

    # Distribuzione della frequenza media
    axs[1].plot(librosa.fft_frequencies(sr=WAVEFORM_SR), np.mean(freqs, axis=0))
    axs[1].set_title(f"Distribuzione della frequenza media per {label.capitalize()}")
    axs[1].set_xlabel("Frequenza (Hz)")
    axs[1].set_ylabel("Ampiezza media")

    plt.tight_layout()
    plt.show()

# Funzione per visualizzare insieme (side by side) la waveform e la heatmap degli MFCC per un file audio specifico
def plot_waveform_and_mfcc(file_path, label):
    # Carica l'audio
    audio, sr = librosa.load(file_path, sr=None)
    # Calcola gli MFCC
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))

    # Waveform
    librosa.display.waveshow(audio, sr=sr, ax=axs[0])
    axs[0].set_title(f"Waveform per {label.capitalize()}")
    axs[0].set_xlabel("Tempo (s)")
    axs[0].set_ylabel("Ampiezza")

    # Heatmap degli MFCC
    img = librosa.display.specshow(mfccs, sr=sr, x_axis='time', cmap='coolwarm', ax=axs[1])
    axs[1].set_title(f"Heatmap degli MFCC per {label.capitalize()}")
    axs[1].set_xlabel("Tempo (s)")
    axs[1].set_ylabel("Coefficiente MFCC")
    fig.colorbar(img, ax=axs[1], format="%+2.0f dB")

    plt.tight_layout()
    plt.show()

###Applicazione delle funzioni di analisi
Qui vengono applicate le funzioni, mostrando i relativi grafici per ogni specie

In [ ]:
# plotta durata distr. durate e frequenze degli audio
def analyze_species_data(data_paths):
    for label, files in data_paths.items():
        if files:
            durations = get_audio_durations(files)
            mean_freqs = get_frequency_distribution(files)
            plot_duration_and_frequency(durations, mean_freqs, label)

# Funzione per analizzare un singolo file audio mostrando waveform e heatmap degli MFCC in un'unica figura
def analyze_audio(file_path, label):
    plot_waveform_and_mfcc(file_path, label)

# Esempio di utilizzo:
# Analisi aggregata sui dati di training: per ogni categoria, vengono visualizzate distribuzione delle durate e frequenze insieme
analyze_species_data(data_paths)

# Analisi di un audio specifico (ad esempio, il primo file di ogni categoria)
for label, files in data_paths.items():
    if files:
        analyze_audio(files[0], label)

# **Modello su Forma D'onda**
Utilizzeremo un modello ibrido, basato su una CNN 1D e uno strato LSTM per analizzare direttamente le sequenze temporali della forma d’onda.

##**Funzioni di data augmentation**
Per migliorare la robustezza del modello e la sua capacità di generalizzare, vengono applicate diverse tecniche di data augmentation.

In [ ]:


#aggiunge rumore bianco (utile per simulare ambienti rumorosi)
def add_white_noise(audio, noise_level_min=0.005, noise_level_max=0.03):
    noise_level = np.random.uniform(noise_level_min, noise_level_max)
    noise = np.random.randn(len(audio))
    return np.clip(audio + noise_level * noise, -1.0, 1.0)

#fa stretch dell'audio (modifica velocità senza cambiare intonazione)
def my_time_stretch(audio, rate):
    if audio.ndim > 1:
        audio = audio[:, 0]
    return librosa.effects.time_stretch(audio,rate = rate)

#modifica l'intonazione(utilissimo per creare varianti di tono per gatti o cani)
def pitch_shift(audio,sr, n_steps):
    if audio.ndim > 1:
        audio = audio[:, 0]
    return librosa.effects.pitch_shift(audio, sr = sr, n_steps=n_steps)


def preemphasis_harmonic(audio, sr):
    audio = librosa.effects.preemphasis(audio, coef=0.97)
    return librosa.effects.hpss(audio)[0]


def reverse_audio(audio, sr):
    return audio[::-1]

#cambia volume
def random_volume(audio, sr, min_gain=0.6, max_gain=1.5):
    gain = np.random.uniform(min_gain, max_gain)
    return np.clip(audio * gain, -1.0, 1.0)


#simula trillo dei gatti
def trill_effect(audio, sr):
    n = len(audio)
    tremolo = np.sin(2 * np.pi * 15 * np.arange(n) / sr)
    return np.clip(audio * (1 + 0.3 * tremolo), -1.0, 1.0)

# Incrementa il guadagno lungo la durata del segnale.
def temporal_gain_ramp(audio, sr):
    n = len(audio)
    gain_ramp = np.linspace(1, 3, n)
    return np.clip(audio * gain_ramp, -1.0, 1.0)

#applica un effetto riverbero sommando un'eco ritardata
def dog_reverb_effect(audio, sr, reverb_level=0.2):
    delay = int(0.03 * sr)  # 30ms di delay
    echo = np.pad(audio, (delay, 0), mode='constant')[:len(audio)]
    return np.clip(audio + reverb_level * echo, -1.0, 1.0)





##**Preprocessing dei dati audio**
I dati vengono caricati, normalizzati e processati per garantire una forma uniforme.  
Le augmentazioni vengono applicate con probabilità casuali per arricchire il dataset.  
Dopo il preprocessing, i dati vengono suddivisi in training e validation set mantenendo il bilanciamento delle classi.

In [ ]:
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ------------------------------
# Funzioni PREPOCESSAMENTO e AUGMENTAZIONE audio
# ------------------------------

def preprocess_audio(file_path, target_length=WAVEFORM_SAMPLES, augment_fn=None):
    """
    Carica e normalizza un file audio, applicando eventuali augmentazioni.

    - Ricampiona a WAVEFORM_SR prima delle trasformazioni.
    - Se sono fornite funzioni di augmentazione, le applica (50% di probabilità ciascuna).
    - Adatta la lunghezza: tronca se troppo lungo, aggiunge padding se troppo corto.

    Argomenti:
        file_path (str): Percorso del file audio.
        target_length (int): Numero di campioni desiderato (default WAVEFORM_SAMPLES).
        augment_fn (list, opzionale): Lista di funzioni.

    Ritorno:
        np.array: L'audio preprocessato.
    """
    # carica file
    y, sr = librosa.load(file_path, sr=WAVEFORM_SR)

    # applica f. augmentazione
    if augment_fn is not None:
        for fn in augment_fn:
            if np.random.rand() > 0.5:
                y = fn(y, sr)

    # adatta la lunghezza
    if len(y) > target_length:
        y = y[:target_length]
    else:
        y = np.pad(y, (0, target_length - len(y)), mode="constant")

    return y


def apply_random_augmentation(audio, sr, label, augmentations_with_prob):
    """
    Applica delle augmentazioni all'audio

    Args:
        audio (np.array): Campione da modificare.
        sr (int): Frequenza di campionamento.
        label (int): Etichetta del campione.
        augmentations_with_prob (list): Lista funzioni di augmentazione.

    Ritorno:
        np.array: Audio eventualmente modificato.
    """
    for fn, prob, apply_to in augmentations_with_prob:
        # applica solo se l'etichetta è tra quelle specificate
        if label in apply_to and np.random.rand() < prob:
            audio = fn(audio, sr)
    return audio



# ------------------------------
# Creazione DATASET
# ------------------------------

X = []   # audio training
y = []   # etichette training
X_test = []  # audio test
y_test = []  # etichette test

label_to_index = {"cat": 0, "dog": 1, "dolphin": 2}

# carica dati test
for label, files in test_data_paths.items():
    for file_path in files:
        X_test.append(preprocess_audio(file_path))
        y_test.append(label_to_index[label])

# carica dati training
for label, files in data_paths.items():
    for file_path in files:
        X.append(preprocess_audio(file_path))
        y.append(label_to_index[label])

# convertiamo il test set in array NumPy e aggiungiamo una dimensione extra (canale)
X_test = np.expand_dims(np.array(X_test, dtype='float32'), axis=-1)
y_test = np.array(y_test)

# ------------------------------
# Suddivisione TRAINING E TEST SET
# ------------------------------

# divisione 80/20 mantenendo bilanciamento
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VALIDATION_FRACTION, random_state=SEED, stratify=y
)

# ------------------------------
# Lista AUGMENTAZIONI
# ------------------------------

# contiene tutte le funzioni di augmentazioni, specifiche per classi. Sono associate a una probabilità che vengano applicate
augmentations_with_prob = [
    (lambda x, sr: add_white_noise(x, noise_level_min=0.001, noise_level_max=0.02), 0.6, [0, 1, 2]),
    (lambda x, sr: my_time_stretch(x, np.random.uniform(0.85, 1.15)), 0.4, [0, 1]),
    (lambda x, sr: pitch_shift(x, sr, np.random.randint(-3, 3)), 0.5, [0]),
    (lambda x, sr: reverse_audio(x, sr), 0.1, [0]),
    (lambda x, sr: random_volume(x, sr, min_gain=0.7, max_gain=1.3), 0.5, [0, 1, 2]),
    (preemphasis_harmonic, 0.6, [0]),
    (trill_effect, 0.4, [0]),
    (lambda x, sr: pitch_shift(x, sr, np.random.choice([-1, 1])), 0.5, [0]),
    (temporal_gain_ramp, 0.7, [0]),
    (dog_reverb_effect, 0.5, [1])
]

# ------------------------------
# Creazione del TRAINING SET AUGMENTATO
# ------------------------------

X_train_aug = []
y_train_aug = []

# ciclo su tutti gli elementi del training
for i in range(len(X_train)):
    X_train_aug.append(X_train[i])
    y_train_aug.append(y_train[i])

    # applicazione delle augmentazioni
    num_aug = 2 if y_train[i] == 1 else 1   # 2 augmentazioni solo per cani
    for _ in range(num_aug):
        augmented_audio = apply_random_augmentation(X_train[i].copy(), WAVEFORM_SR, y_train[i], augmentations_with_prob)
        # Forza la lunghezza a WAVEFORM_SAMPLES campioni: se troppo lungo, tronca; se troppo corto, aggiungi padding.
        if len(augmented_audio) > WAVEFORM_SAMPLES:
            augmented_audio = augmented_audio[:WAVEFORM_SAMPLES]
        else:
            augmented_audio = np.pad(augmented_audio, (0, WAVEFORM_SAMPLES - len(augmented_audio)), mode="constant")
        X_train_aug.append(augmented_audio)
        y_train_aug.append(y_train[i])


# conversione in array NumPy + aggiunta canali
X_train_aug = np.expand_dims(np.array(X_train_aug, dtype='float32'), axis=-1)
y_train_aug = np.array(y_train_aug)
X_val = np.expand_dims(np.array(X_val, dtype='float32'), axis=-1)

assert len(X_train_aug) == len(y_train_aug), "Mismatch tra X_train_aug e y_train_aug!"

# ------------------------------
# NORMALIZZAZIONE
# ------------------------------

mean = np.mean(X_train_aug)
std = max(float(np.std(X_train_aug)), np.finfo(np.float32).eps)
X_train_aug = (X_train_aug - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std



###**Comparazione dati originali/augmentati**
Qui vengono graficati il numero di campionamento prima e dopo l'applicazione di augmentazione

In [ ]:
import matplotlib.pyplot as plt

# conteggio
original_counts = [y_train.count(i) for i in range(len(label_to_index))]
augmented_counts = np.bincount(y_train_aug)

labels = list(label_to_index.keys())
x = np.arange(len(labels))

# fa il plotting
fig, ax = plt.subplots(figsize=(10, 6))

# barra dati originali
bar_width = 0.35
ax.bar(x - bar_width/2, original_counts, bar_width, label='Originali', color='skyblue')

# barra dati augmentati
ax.bar(x + bar_width/2, augmented_counts, bar_width, label='Augmentati', color='orange')

# setup grafico
ax.set_xlabel('Categoria')
ax.set_ylabel('Numero di esempi')
ax.set_title('Numero di esempi per categoria: Originale vs Augmentato')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# visualizza grafico
plt.tight_layout()
plt.show()


##**Costruzione Modello Forma D'onda**
Si tratta di un modello ibrido **CNN-LSTM**:

**Conv1D** con kernel crescenti per estrarre caratteristiche temporali a diverse scale.

**MaxPooling** per ridurre la dimensionalità e migliorare la robustezza.

**Batch Normalization** e **Dropout** per stabilizzare il training e prevenire overfitting.

**Bidirectional LSTM** per catturare la dinamica temporale nei segnali audio.

**Global Average Pooling** e **Global Max Pooling** per sintetizzare le feature estratte.

**Dense layers** con regularization per l'output finale con attivazione softmax.

Il modello viene ottimizzato con AdamW, utilizzando sparse categorical crossentropy come funzione di perdita e l'accuratezza come metrica principale.


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import Bidirectional, LSTM, GlobalAveragePooling1D, GlobalMaxPooling1D, Concatenate, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import AdamW

#input
input_layer = Input(shape=(WAVEFORM_SAMPLES, 1))

#blocco CNN piccolo per catturare a grana fine
x = Conv1D(32, kernel_size=32, activation='relu', padding='same',kernel_regularizer=l2(0.0001))(input_layer)
x = MaxPooling1D(2)(x)
x = Dropout(0.2)(x)

#blocco CNN medio per catturare suoni intermedi (miagolii e abbai)
x = Conv1D(64, kernel_size=64, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(x)
x = MaxPooling1D(4)(x)
x = Dropout(0.3)(x)

#blocco CNN grande per catturare caratteristiche globali
x = Conv1D(128, kernel_size=128, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(x)
x = MaxPooling1D(4)(x)
#normalizzazione per stabilizzare training
x = Dropout(0.4)(x)

#blocco CNN finale molto piccolo per estrarre ulteriori carattestiche locali
x = Conv1D(128, kernel_size=16, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(x)
x = MaxPooling1D(2)(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)



#blocco LSTM per catturare la sequenzialità temporale dei dati
x = Bidirectional(LSTM(64, return_sequences=True))(x)

#sintetizzazione finale delle feature
avg_pool = GlobalAveragePooling1D()(x)
max_pool = GlobalMaxPooling1D()(x)
x = Concatenate()([avg_pool, max_pool])

#strati densi per classificazione finale
x = Dense(128, activation='relu', kernel_regularizer=l2(0.0001))(x)
x = Dropout(0.5)(x)
output_layer = Dense(3, activation='softmax')(x)

#creazione modello
model = Model(inputs=input_layer, outputs=output_layer)
#compilazione modello
model.compile(
    optimizer=AdamW(learning_rate=0.0001, weight_decay=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Addestramento del modello waveform

Uso pesi di classe bilanciati, batch da 16 e un massimo di `MAX_EPOCHS` epoche. Early stopping monitora la loss di validazione con pazienza 8; ReduceLROnPlateau dimezza il learning rate dopo 3 epoche senza miglioramenti. I pesi della migliore epoca vengono ripristinati.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import collections

#calcolo pesi
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

y_val = np.array(y_val, dtype=np.int32)

#callback
early_stopping = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)


#addestramento
history = model.fit(
    X_train_aug, y_train_aug,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=16,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weights
)



#salviamo il modello
model.save(str(ARTIFACT_DIR / "audio_waveform_model.keras"))


##**Performance**

###**Accuracy e Loss**
L'andamento dell'addestramento viene visualizzato con due grafici, uno per l'andamento della **Accuracy** e uno per quello della *Loss**. In entrambi si tiene in considerazione sia di training che di validation

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

#accuracy
ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Andamento dell\'Accuracy')
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

#loss
ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Andamento della Loss')
ax2.set_xlabel('Epoca')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.show()




###**Perfomance su Test Set**
Il modello viene testato sul set di test per valutare le metriche finali

In [ ]:
from tensorflow.keras.models import load_model

model = load_model(str(ARTIFACT_DIR / "audio_waveform_model.keras"))


test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)

print(f"Performance sul set di test:")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")
# predizioni
y_pred = model.predict(X_test)

# assume probabilità più alta come classe predetta
y_pred_classes = np.argmax(y_pred, axis=1)

# mappa numero a classe
index_to_label = {0: "cat", 1: "dog", 2: "dolphin"}
errori_gatto, errori_cane, errori_delfino = 0, 0, 0

# conteggio errori
for i, file_path in enumerate(sum(test_data_paths.values(), [])):
    predicted_label = index_to_label[y_pred_classes[i]]
    actual_label = index_to_label[y_test[i]]
    if predicted_label != actual_label:
        if actual_label == "cat":
            errori_gatto += 1
        elif actual_label == "dog":
            errori_cane += 1
        elif actual_label == "dolphin":
            errori_delfino += 1
print("Numero di errori per ogni classe:")
print(f"Gatto: {errori_gatto}")
print(f"Cane: {errori_cane}")
print(f"Delfino: {errori_delfino}")



###**Confusion Matrix e Metriche**


Qui vengono visualizzate la matrice di confusione e le principali metriche di classificazione (precision, recall, F1-score).

La **matrice di confusione** mostra la distribuzione delle predizioni corrette e degli errori tra le classi.

Le **metriche di classificazione** sono:

**Precision**: frazione di predizioni corrette tra quelle assegnate a una classe    

**Recall**: frazione di campioni di una classe correttamente riconosciuti  

**F1-score**: media armonica tra precision e recall

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

#previsioni su test set e calcolo confusion matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

#confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["cat", "dog", "dolphin"],
            yticklabels=["cat", "dog", "dolphin"], ax=axes[0])
axes[0].set_title('Matrice di Confusione')
axes[0].set_xlabel('Classe Predetta')
axes[0].set_ylabel('Classe Reale')

#grafico metriche
report = classification_report(y_test, y_pred, target_names=["cat", "dog", "dolphin"], output_dict=True)
classes = ["cat", "dog", "dolphin"]
metrics = ['precision', 'recall', 'f1-score']

data = {m: [report[cls][m] for cls in classes] for m in metrics}
x = np.arange(len(classes))
width = 0.2

axes[1].bar(x - width, data['precision'], width, label='Precision')
axes[1].bar(x, data['recall'], width, label='Recall')
axes[1].bar(x + width, data['f1-score'], width, label='F1-Score')
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Valore')
axes[1].set_title('Metriche per Classe')
axes[1].set_xticks(x)
axes[1].set_xticklabels(classes)
axes[1].legend()
axes[1].grid(True)

#mostra i grafici
plt.tight_layout()
plt.show()



# **Modello Spettrogramma**

##**Funzioni di Augmentazione**
Per migliorare la robustezza del modello e la sua capacità di generalizzare, vengono applicate diverse tecniche di data augmentation.  

In [ ]:
def normalize_audio(y_audio):
    peak = np.max(np.abs(y_audio))
    return np.clip(y_audio / max(float(peak), np.finfo(np.float32).eps), -1.0, 1.0)

#modifica l'intonazione(utilissimo per creare varianti di tono per gatti o cani)
def pitch_augment(y_audio, sr):

    augmenter = PitchShift(min_semitones=-5, max_semitones=5, p=1.0)
    return augmenter(y_audio, sample_rate=sr)

#filtra il segnale mantenendo solo una determinata banda di frequenze(utile per cani)
def bandpass_filter(y_audio, sr):

    augmenter = BandPassFilter(min_center_freq=2000, max_center_freq=5000, p=1.0)
    return augmenter(y_audio, sample_rate=sr)

#introduce distrosione armonica per aumentare la diversità dei dati.
def tanh_distortion(y_audio, sr):
    augmenter = TanhDistortion(min_distortion=0.3, max_distortion=0.7)
    return augmenter(y_audio, sample_rate=sr)

#modifica velocità audio senza alterarne l'intonazione
def time_augment(y_audio, sr):
    augmenter = TimeStretch(min_rate=0.7, max_rate=1.3)
    return augmenter(y_audio, sample_rate=sr)

#modifica il volume per adattarsi a diverse intensità sonore.
def amplitude_augment(y_audio, min_gain=0.8, max_gain=1.5):
    gain = np.random.uniform(min_gain, max_gain)
    return y_audio * gain

    #aggiunge rumore bianco (utile per simulare ambienti rumorosi)
def add_white_noise(audio, noise_level_min=0.005, noise_level_max=0.03):
    noise_level = np.random.uniform(noise_level_min, noise_level_max)
    noise = np.random.randn(len(audio))
    return np.clip(audio + noise_level * noise, -1.0, 1.0)
    #applica un effetto riverbero sommando un'eco ritardata
def dog_reverb_effect(audio, sr, reverb_level=0.2):
    delay = int(0.03 * sr)  # 30ms di delay
    echo = np.pad(audio, (delay, 0), mode='constant')[:len(audio)]
    return np.clip(audio + reverb_level * echo, -1.0, 1.0)



##**Preprocessing dei dati audio**
I dati vengono caricati, normalizzati, processati e convertiti in spettrogramma a una forma uniforme di 128 Mel.  

Le augmentazioni vengono applicate sull'audio Raw con probabilità casuali per arricchire il dataset prima che avvenga la conversione in spettrogramma.

Dopo il preprocessing, i dati vengono suddivisi in training e validation set mantenendo il bilanciamento delle classi.



In [ ]:
import librosa
import numpy as np
from glob import glob
from sklearn.model_selection import train_test_split

from sklearn.utils import shuffle
import collections

#----------------------------------------------
# Parametri di CONFIGURAZIONE
#----------------------------------------------
SR = 22050                    # Sample rate
DURATION = 4.0                # Durata target in secondi
N_FFT = 2048                  # FFT window size
HOP_LENGTH = 512              # FFT hop length
N_MELS = 128                  # Numero di bande Mel
SPEC_SHAPE = (N_MELS, int(DURATION * SR / HOP_LENGTH))  # Forma finale spettrogrammi

#----------------------------------------------
# Funzione conversione AUDIO --> SPETTROGRAMMA
#----------------------------------------------
def audio_to_melspectrogram(y, sr=SR):
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
        fmin=20, fmax=sr // 2
    )
    S_dB = librosa.power_to_db(S, ref=np.max)
    # Tronca o pad per avere SPEC_SHAPE[1] frame temporali
    if S_dB.shape[1] > SPEC_SHAPE[1]:
        S_dB = S_dB[:, :SPEC_SHAPE[1]]
    else:
        pad_width = [(0, 0), (0, SPEC_SHAPE[1] - S_dB.shape[1])]
        S_dB = np.pad(S_dB, pad_width, mode='constant')
    return S_dB

#----------------------------------------------
# Funzione di AUGMENTAZIONE sull'AUDIO RAW
#----------------------------------------------
def apply_augmentation_audio(y_audio, label, intensity=1, sr=SR):
    y_audio = y_audio.astype(np.float32)
    original_audio = y_audio.copy()

    if label == 0:  # Gatto
        if np.random.rand() < 0.3 * intensity:
            y_audio = normalize_audio(y_audio)
            y_audio = pitch_augment(y_audio, sr)
    elif label == 1:  # Cane
        if np.random.rand() < 0.6 * intensity:
            y_audio = normalize_audio(y_audio)
            y_audio = bandpass_filter(y_audio, sr)
        if np.random.rand() < 0.3 * intensity:
            y_audio = time_augment(y_audio, sr)
        if np.random.rand() < 0.4 * intensity:
            y_audio = pitch_augment(y_audio, sr)
        if np.random.rand() < 0.7 * intensity:
            y_audio = amplitude_augment(y_audio, min_gain=0.8, max_gain=1.5)
        if np.random.rand() < 0.3 *intensity :
            y_audio = dog_reverb_effect(y_audio, sr)
    return y_audio
#----------------------------------------------
# Funzione per CARICARE I FILE AUDIO RAW (senza trasformarli in spettrogramma)
#----------------------------------------------
def load_raw_dataset(paths_dict):
    """
    Carica i file audio da paths_dict senza applicare alcuna augmentazione o trasformazione.
    Restituisce due liste: una con i segnali audio raw e una con le etichette.

    paths_dict: dizionario {label: [lista_di_file_audio]}
    """
    audio_list = []
    label_list = []
    label_to_index = {"cat": 0, "dog": 1, "dolphin": 2}

    for label, files in paths_dict.items():
        for file_path in files:
            y_audio, sr = librosa.load(file_path, sr=SR)
            target_samples = int(SR * DURATION)
            if len(y_audio) > target_samples:
                y_audio = y_audio[:target_samples]
            else:
                y_audio = np.pad(y_audio, (0, target_samples - len(y_audio)), mode='constant')
            audio_list.append(y_audio)
            label_list.append(label_to_index[label])
    return np.array(audio_list), np.array(label_list)



#----------------------------------------------
# 1. CARICAMENTO DEI DATI RAW (SENZA AUGMENTAZIONE)
#----------------------------------------------
X_raw, y_raw = load_raw_dataset(data_paths)
X_test_raw, y_test = load_raw_dataset(test_data_paths)

#----------------------------------------------
# 2. SPLIT: DIVIDIAMO I DATI RAW IN TRAINING E VALIDATION
#----------------------------------------------
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y_raw,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=y_raw
)
# Ora X_val_raw conterrà i segnali audio raw da trasformare direttamente in spettrogrammi (senza augmentation)

#----------------------------------------------
# 3. CREAZIONE DEI DATI VALIDATION: CONVERSIONE IN SPETTROGRAMMI (SENZA AUGMENTAZIONE)
#----------------------------------------------
X_val = []
for y_audio in X_val_raw:
    spec = audio_to_melspectrogram(y_audio, sr=SR)
    X_val.append(spec)
X_val = np.array(X_val)

#----------------------------------------------
# 4. CREAZIONE DEI DATI TRAINING: AUGMENTAZIONE E CONVERSIONE IN SPETTROGRAMMI
#----------------------------------------------
# Definiamo i fattori di augmentation per ciascuna classe:
augmentation_factors = {0: 1, 1: 2, 2: 1}

X_train_list = []
y_train_list = []

# Per ogni esempio del training set raw, salviamo anche la versione originale e le copie augmentate
for idx, y_audio in enumerate(X_train_raw):
    # Salviamo la versione originale
    spec = audio_to_melspectrogram(y_audio, sr=SR)
    X_train_list.append(spec)
    y_train_list.append(y_train[idx])

    # Recuperiamo il fattore di augmentation in base all'etichetta
    factor = augmentation_factors[y_train[idx]]
    for _ in range(factor):
        # Applichiamo l'augmentation partendo dalla copia del segnale originale
        augmented_audio = apply_augmentation_audio(y_audio.copy(), y_train[idx], intensity=1, sr=SR)
        augmented_spec = audio_to_melspectrogram(augmented_audio, sr=SR)
        X_train_list.append(augmented_spec)
        y_train_list.append(y_train[idx])

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)

#----------------------------------------------
# 5. NORMALIZZAZIONE E AGGIUNTA DELLA DIMENSIONE CANALE
#----------------------------------------------
# Aggiungiamo la dimensione canale agli array di spettrogrammi
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = []
# Per X_test, lavoriamo sui dati raw di test e li convertiamo senza augmentazione
for y_audio in X_test_raw:
    spec = audio_to_melspectrogram(y_audio, sr=SR)
    X_test.append(spec)
X_test = np.array(X_test)
X_test = X_test[..., np.newaxis]

# Shuffling del training set
X_train, y_train = shuffle(X_train, y_train, random_state=SEED)

# Calcola mean e std sul training set per normalizzare anche validation e test
mean = X_train.mean()
std = max(float(X_train.std()), np.finfo(np.float32).eps)
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std



###Comparazione dati originali/augmentati
Qui vengono graficati il numero di campionamento prima e dopo l'applicazione di augmentazione

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

label_to_index = {"cat": 0, "dog": 1, "dolphin": 2}

# Conta esempi
original_counts = np.bincount(y_raw)
augmented_counts = np.bincount(y_train)

# Setup grafico
labels = list(label_to_index.keys())
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.35

# Barra dati originali
ax.bar(x - bar_width / 2, original_counts, bar_width, label='Originali', color='skyblue')

# Barra dati augmentati
ax.bar(x + bar_width / 2, augmented_counts, bar_width, label='Augmentati', color='orange')

# Setup
ax.set_xlabel('Categoria')
ax.set_ylabel('Numero di esempi')
ax.set_title('Numero di esempi per categoria: Originale vs Augmentato')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

plt.tight_layout()
plt.show()


***

###**Visualizzazione spettrogrammi originali/modificati**
Qui viene visualizzati gli spettogrammi di un campione prima e dopo l'augmentazione

In [ ]:
import matplotlib.pyplot as plt

# Seleziona un esempio casuale
idx = np.random.randint(len(X_raw))

# Seleziona l'audio originale
original_audio = X_raw[idx]

# Crea lo spettrogramma dell'audio originale
original_spec = audio_to_melspectrogram(original_audio, sr=SR)

# Applica le trasformazioni previste per la classe del campione.
augmented_audio = apply_augmentation_audio(original_audio.copy(), y_raw[idx], intensity=1, sr=SR)

# Crea lo spettrogramma dell'audio augmentato
augmented_spec = audio_to_melspectrogram(augmented_audio, sr=SR)

# Visualizza gli spettrogrammi
plt.figure(figsize=(10, 4))

# Spettrogramma originale
plt.subplot(1, 2, 1)
plt.imshow(original_spec, aspect='auto', origin='lower')
plt.title("Originale")

# Spettrogramma augmentato
plt.subplot(1, 2, 2)
plt.imshow(augmented_spec, aspect='auto', origin='lower')
plt.title("Augmentato")

plt.tight_layout()
plt.show()


##**Costruzione Modello Spettrogramma**
Il modello è una CNN progettata per analizzare le caratteristiche spaziali del suono:

**Input Layer**: Accetta spettrogrammi di dimensione (128, 172, 1).

**Conv2D** (64 filtri, kernel 5x25): Estrae pattern iniziali a livello temporale.

**MaxPooling2D** (2x4): Riduce la dimensionalità e migliora la robustezza alle variazioni.

**Conv2D** (128 filtri, kernel 25x5): Estrae pattern più complessi combinando frequenza e tempo.

**MaxPooling2D** (4x2): Riduce ulteriormente la dimensionalità mantenendo le informazioni più rilevanti.

**Conv2D** (256 filtri, kernel 15x15): Cattura relazioni tra frequenze e dinamica del suono.

**Batch Normalization** e **Dropout** stabilizzano il training e previene rischio di overfitting.

**Global Average Pooling 2D** converte le feature maps in un vettore di feature compatto.

**Dense Layer** (512 unità, ReLU) con regularization per l'output finale con attivazione softmax

L'ottimizzazione viene effettuata con AdamW, con sparse categorical crossentropy come funzione di perdita e accuratezza come metrica principale.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import AdamW

# funzione per creare il modello con input specifico
def create_model(input_shape=(128, 172, 1)):
    model = models.Sequential([

        layers.Input(shape=input_shape),

        # primo blocco CNN per estrarre caratteristiche iniziali
        layers.Conv2D(64, (5,25), activation='relu', padding='same', kernel_regularizer=l2(0.0001)),
        layers.MaxPooling2D((2,4)),

        # secondo blocco CNN per catturare caratteristiche intermedie
        layers.Conv2D(128, (25,5), activation='relu', padding='same',kernel_regularizer=l2(0.0001)),
        layers.MaxPooling2D((4,2)),

        # terzo blocco CNN per catturare caratteristiche globali
        layers.Conv2D(256, (15,15), activation='relu', padding='same', kernel_regularizer=l2(0.0001)),
        # normalizzazione per stabilizzare il training

        layers.MaxPooling2D((3,3)),
        # pooling globale per ridurre la dimensionalità

        layers.Conv2D(256, (7,7), activation='relu', padding='same', kernel_regularizer=l2(0.0001)),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),

        # strati densi per la classificazione finale
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),  # regolarizzazione tramite dropout
        layers.Dense(3, activation='softmax')  # output con 3 classi
    ])
    return model

# creazione modello
input_shape = (128, 172, 1)
model2 = create_model(input_shape)

# compilazione del modello con ottimizzatore AdamW
model2.compile(
    optimizer=AdamW(learning_rate=0.0001, weight_decay=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


model2.summary()


## Addestramento del modello Mel

Uso pesi bilanciati e moltiplico quello della classe cane per `DOG_CLASS_WEIGHT_MULTIPLIER`, pari a 5 nell'esperimento originale. Questa scelta richiede una nuova verifica su dati reali. Il batch contiene 16 esempi e il limite è `MAX_EPOCHS`.

Early stopping monitora la loss di validazione con pazienza 10 e ripristina i migliori pesi. ReduceLROnPlateau monitora l'accuracy di validazione con pazienza 3 e fattore 0.3. Il checkpoint viene scritto nella cartella degli output.


In [ ]:

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import collections
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

print(collections.Counter(y_train))
class_weights[1] *= DOG_CLASS_WEIGHT_MULTIPLIER

y_val = np.array(y_val, dtype=np.int32)

# callbacks
early_stopping = EarlyStopping(monitor='val_loss',
                               patience=10,
                               restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(
    monitor="val_accuracy",
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)
model_checkpoint = ModelCheckpoint(str(ARTIFACT_DIR / "best_model.keras"), monitor='val_loss', save_best_only=True)



# Addestramento con EarlyStopping
history2 = model2.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=MAX_EPOCHS,
                    batch_size=16,
                    callbacks=[early_stopping,reduce_lr,model_checkpoint],
                    shuffle=True,
                    class_weight=class_weights
                    )



# Salviamo il modello
model2.save(str(ARTIFACT_DIR / "spectrogram_model.keras"))


##**Performance**

###**Accuracy e Loss**
L'andamento dell'addestramento viene visualizzato con due grafici, uno per l'andamento della **Accuracy** e uno per quello della *Loss**. In entrambi si tiene in considerazione sia di training che di validation

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

#accuracy
ax1.plot(history2.history['accuracy'], label='Train Accuracy')
ax1.plot(history2.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Andamento dell\'Accuracy')
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

#loss
ax2.plot(history2.history['loss'], label='Train Loss')
ax2.plot(history2.history['val_loss'], label='Validation Loss')
ax2.set_title('Andamento della Loss')
ax2.set_xlabel('Epoca')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.show()




###**Perfomance su Test Set**
Il modello viene testato sul set di test per valutare le metriche finali

In [ ]:
from tensorflow.keras.models import load_model

model2 = load_model(str(ARTIFACT_DIR / "spectrogram_model.keras"))


test_loss, test_accuracy = model2.evaluate(X_test, y_test, verbose=1)

print(f"Performance sul set di test:")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")
# predizioni
y_pred = model2.predict(X_test)

# assume probabilità più alta come classe predetta
y_pred_classes = np.argmax(y_pred, axis=1)

# mappa numero a classe
index_to_label = {0: "cat", 1: "dog", 2: "dolphin"}
errori_gatto, errori_cane, errori_delfino = 0, 0, 0

# conteggio errori
for i, file_path in enumerate(sum(test_data_paths.values(), [])):
    predicted_label = index_to_label[y_pred_classes[i]]
    actual_label = index_to_label[y_test[i]]
    if predicted_label != actual_label:

        if actual_label == "cat":
            errori_gatto += 1
        elif actual_label == "dog":
            errori_cane += 1
        elif actual_label == "dolphin":
            errori_delfino += 1
print("Numero di errori per ogni classe:")
print(f"Gatto: {errori_gatto}")
print(f"Cane: {errori_cane}")
print(f"Delfino: {errori_delfino}")



###**Confusion Matrix e Metriche**
Qui vengono visualizzate la matrice di confusione e le principali metriche di classificazione (precision, recall, F1-score).

La **matrice di confusione** mostra la distribuzione delle predizioni corrette e degli errori tra le classi.

Le **metriche di classificazione** sono:

**Precision**: frazione di predizioni corrette tra quelle assegnate a una classe    

**Recall**: frazione di campioni di una classe correttamente riconosciuti  

**F1-score**: media armonica tra precision e recall

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns


y_pred = np.argmax(model2.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

#confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["cat", "dog", "dolphin"],
            yticklabels=["cat", "dog", "dolphin"], ax=axes[0])
axes[0].set_title('Matrice di Confusione')
axes[0].set_xlabel('Classe Predetta')
axes[0].set_ylabel('Classe Reale')

#grafico metriche
report = classification_report(y_test, y_pred, target_names=["cat", "dog", "dolphin"], output_dict=True)
classes = ["cat", "dog", "dolphin"]
metrics = ['precision', 'recall', 'f1-score']


data = {m: [report[cls][m] for cls in classes] for m in metrics}

x = np.arange(len(classes))
width = 0.2

axes[1].bar(x - width, data['precision'], width, label='Precision')
axes[1].bar(x, data['recall'], width, label='Recall')
axes[1].bar(x + width, data['f1-score'], width, label='F1-Score')
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Valore')
axes[1].set_title('Metriche per Classe')
axes[1].set_xticks(x)
axes[1].set_xticklabels(classes)
axes[1].legend()
axes[1].grid(True)

#mostra i grafici
plt.tight_layout()
plt.show()
